In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch
import gc
from torch.nn.functional import softmax
from sklearn.utils.class_weight import compute_class_weight
from torch.nn import CrossEntropyLoss
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Load the dataset

df = pd.read_excel("data.xlsx") 
df['category'] = df['category'].str.strip()
print("Categories:", df["category"].unique())

# Split Train/Test 
X_train, X_test, y_train, y_test = train_test_split(
    df['title'], df['category'], test_size=0.3, random_state=42
)

In [ ]:
# Group rare classes

counts = df['category'].value_counts()
frequent_classes = counts[counts >= 2].index
df['category'] = df['category'].where(df['category'].isin(frequent_classes), other='other')

In [ ]:
# Simple cleaning: lowercase + remove special characters + strip spaces

df['cleaned_title'] = (
    df['title']
    .str.lower()
    .str.replace(r'[^a-z0-9\s]', '', regex=True)
    .str.strip()
)

In [ ]:
# Encode the labels on the whole dataset

le = LabelEncoder()
df['category_enc'] = le.fit_transform(df['category'])
df['category'] = df['category'].replace('other', 'unclassified')

# Remove the rows where the category is "unclassified"
df= df[df['category'] != "unclassified"]

In [ ]:
print(df['category'].value_counts())

In [ ]:
# Stratified split on the encoded labels

X_train, X_test, y_train_enc, y_test_enc = train_test_split(
    df['cleaned_title'], df['category_enc'],
    test_size=0.2,
    random_state=42,
    stratify=df['category_enc']
)
print("Train distribution:", pd.Series(y_train_enc).value_counts())
print("Test distribution:", pd.Series(y_test_enc).value_counts())

In [ ]:
# DataFrames train/test
train_df = pd.DataFrame({'title': X_train, 'label': y_train_enc})
test_df  = pd.DataFrame({'title': X_test,  'label': y_test_enc})

# no NaNs
train_df['title'] = train_df['title'].astype(str).fillna('')
test_df['title']  = test_df['title'].astype(str).fillna('')

In [ ]:
# HuggingFace Dataset

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

In [ ]:
# Tokenizer

model_name = "camembert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def tokenize(batch):
    # batch['title'] is always a list of strings
    texts = batch['title']
    if isinstance(texts, str):
        texts = [texts]
    else:
        texts = list(texts)
    return tokenizer(texts, truncation=True, padding="max_length", max_length=64)


train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
# Compute the class weights

classes = np.unique(y_train_enc)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train_enc)
class_weights = torch.tensor(weights, dtype=torch.float)

In [ ]:
# Model

num_labels = len(le.classes_)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

In [ ]:
# Redefine the loss with class weights

def custom_compute_loss(model, inputs, return_outputs=False):
    labels = inputs.get("labels")
    outputs = model(**inputs)
    logits = outputs.logits
    loss_fct = CrossEntropyLoss(weight=class_weights.to(logits.device))
    loss = loss_fct(logits, labels)
    return (loss, outputs) if return_outputs else loss

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=10,
    per_device_eval_batch_size=10,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_steps=50,
    report_to=None ,
    seed=42,
)

In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=lambda p: {"accuracy": (np.argmax(p.predictions, axis=1) == p.label_ids).mean()},
    class_weights=class_weights
)

In [ ]:
# Training

trainer.train()

In [ ]:
model.save_pretrained("./my_model")
tokenizer.save_pretrained("./my_model")

In [ ]:
# Predictions

preds_output = trainer.predict(test_dataset)
preds = np.argmax(preds_output.predictions, axis=1)
labels = preds_output.label_ids


# Overall metrics

acc = accuracy_score(labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')

print(f"Accuracy   : {acc:.4f}")
print(f"Precision  : {precision:.4f}")
print(f"Recall     : {recall:.4f}")
print(f"F1-score   : {f1:.4f}")



class_names = le.classes_
cm = confusion_matrix(labels, preds)
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)

# plot
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues", xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predictions")
plt.ylabel("Actual")
plt.title("Confusion matrix")
plt.show()

In [ ]:
# Predictions

preds_output = trainer.predict(test_dataset)
preds = np.argmax(preds_output.predictions, axis=1)
labels = preds_output.label_ids

# Detailed report per class

report = classification_report(labels, preds, target_names=le.classes_)
print(report)

In [ ]:
import matplotlib.pyplot as plt

# Data
epochs = [1, 2, 3]
training_loss = [0.456100, 0.371300, 0.275800]
validation_loss = [0.454787, 0.416114, 0.406553]
accuracy = [0.862494, 0.876787, 0.877279]

# Create the figure
fig, ax1 = plt.subplots(figsize=(8,5))

# Loss curves
ax1.plot(epochs, training_loss, label='Training Loss', marker='o')
ax1.plot(epochs, validation_loss, label='Validation Loss', marker='o')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_xticks(epochs)
ax1.legend(loc='upper right')
ax1.grid(True)

# Accuracy curve on the secondary axis
ax2 = ax1.twinx()
ax2.plot(epochs, accuracy, label='Accuracy', color='green', marker='x')
ax2.set_ylabel('Accuracy')
ax2.legend(loc='lower right')

plt.title("Training/Validation Loss and Accuracy over epochs")
plt.show()


In [ ]:
trainer.model.save_pretrained("my_model")